In [ ]:
!pip install ultralytics supervision opencv-python yt-dlp -q

In [ ]:
#EL CLASSICO CLIP 
!yt-dlp "https://youtu.be/9V2guLT3S14?si=BUVQxqpMQnmlI21A" -o test_clip.mp4

In [ ]:
"""
cv_pipeline/detect.py
---------------------
Step 1: Run YOLOv8 detection + ByteTrack on a video clip.
Outputs an annotated video and a CSV of per-frame player positions.

Owner: P1 (cv-pipeline)
"""

import cv2
import csv
import supervision as sv
from ultralytics import YOLO
from pathlib import Path


MODEL_PATH = "yolov8x.pt"  # swap with football-specific weights when ready
SOURCE_VIDEO = "data/samples/test_clip.mp4"
OUTPUT_VIDEO = "outputs/annotated.mp4"
OUTPUT_CSV   = "outputs/positions.csv"


def run(source: str = SOURCE_VIDEO, output_video: str = OUTPUT_VIDEO, output_csv: str = OUTPUT_CSV):
    Path("outputs").mkdir(exist_ok=True)

    model = YOLO(MODEL_PATH)
    tracker = sv.ByteTrack()
    box_annotator = sv.BoxAnnotator()
    label_annotator = sv.LabelAnnotator()

    cap = cv2.VideoCapture(source)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    with open(output_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["frame", "tracker_id", "x_center", "y_center", "confidence", "class"])

        frame_idx = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            results = model(frame, verbose=False)[0]
            detections = sv.Detections.from_ultralytics(results)

            # Filter to persons only (class 0 in COCO)
            detections = detections[detections.class_id == 0]
            detections = tracker.update_with_detections(detections)

            # Write positions
            for i, tracker_id in enumerate(detections.tracker_id):
                x1, y1, x2, y2 = detections.xyxy[i]
                cx = (x1 + x2) / 2
                cy = (y1 + y2) / 2
                writer.writerow([frame_idx, tracker_id, round(cx, 2), round(cy, 2),
                                  round(float(detections.confidence[i]), 3),
                                  detections.class_id[i]])

            # Annotate frame
            labels = [f"#{tid}" for tid in detections.tracker_id]
            frame = box_annotator.annotate(frame, detections)
            frame = label_annotator.annotate(frame, detections, labels)
            out.write(frame)
            frame_idx += 1

    cap.release()
    out.release()
    print(f"Done. Annotated video: {output_video}")
    print(f"Positions CSV: {output_csv}")


if __name__ == "__main__":
    run()


In [ ]:
# change source path to Colab path
run(source="test_clip.mp4", output_video="annotated.mp4", output_csv="positions.csv")

In [ ]:
#problem 0 detections
import pandas as pd
df = pd.read_csv("positions.csv")
print(df.head(20))
print(f"Total detections: {len(df)}")
print(f"Unique players tracked: {df['tracker_id'].nunique()}")

In [ ]:
#problem couldnt encounter frames
import cv2
import csv
import supervision as sv
from ultralytics import YOLO
from pathlib import Path

MODEL_PATH = "yolov8x.pt"
SOURCE_VIDEO = "test_clip.mp4"
OUTPUT_VIDEO = "outputs/annotated.mp4"
OUTPUT_CSV   = "outputs/positions.csv"

Path("outputs").mkdir(exist_ok=True)

model = YOLO(MODEL_PATH)
tracker = sv.ByteTrack()  
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

cap = cv2.VideoCapture(SOURCE_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["frame", "tracker_id", "x_center", "y_center", "confidence", "class"])

    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, verbose=False)[0]
        detections = sv.Detections.from_ultralytics(results)

        # DEBUG: first 5 frames print all classes
        if frame_idx < 5:
            print(f"Frame {frame_idx}: {len(detections)} detections, classes: {detections.class_id}")

        #person class (0)
        detections = detections[detections.class_id == 0]
        detections = tracker.update_with_detections(detections)

        if frame_idx < 5:
            print(f"  After filter+track: {len(detections)} persons, tracker_ids: {detections.tracker_id}")

        for i in range(len(detections)):
            if detections.tracker_id is None:
                continue
            x1, y1, x2, y2 = detections.xyxy[i]
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2
            writer.writerow([
                frame_idx,
                int(detections.tracker_id[i]),
                round(float(cx), 2),
                round(float(cy), 2),
                round(float(detections.confidence[i]), 3),
                int(detections.class_id[i])
            ])

        labels = [f"#{tid}" for tid in (detections.tracker_id or [])]
        frame = box_annotator.annotate(frame, detections)
        frame = label_annotator.annotate(frame, detections, labels)
        out.write(frame)
        frame_idx += 1

        if frame_idx % 100 == 0:
            print(f"Processed {frame_idx} frames...")

cap.release()
out.release()
print(f"\nDone. Total frames: {frame_idx}")

In [ ]:
#file not found error
import os
print(os.path.getsize("test_clip.mp4"), "bytes")

In [ ]:
#looking for file
import os
print(os.listdir("."))

In [ ]:
#retrying downloading file since couldnt find it
!apt-get install -y ffmpeg -q
!ffmpeg -i test_clip.mp4.webm -c:v libx264 test_clip.mp4 -y

In [ ]:
#remove existing clip and extract video from url
!rm test_clip.mp4 test_clip.mp4.webm 2>/dev/null; echo "cleaned"
!yt-dlp "https://youtu.be/9V2guLT3S14?si=BUVQxqpMQnmlI21A" -f "bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[height<=720]" --merge-output-format mp4 -o test_clip.mp4

In [ ]:
#checking frames how many are there and resolution
import cv2
cap = cv2.VideoCapture("test_clip.mp4")
print("Frames:", cap.get(cv2.CAP_PROP_FRAME_COUNT))
print("Resolution:", cap.get(cv2.CAP_PROP_FRAME_WIDTH), "x", cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

In [ ]:
#detecting players using sv yolo and saving it to path
import cv2
import csv
import supervision as sv
from ultralytics import YOLO
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)

model = YOLO("yolov8x.pt")
tracker = sv.ByteTrack()
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

cap = cv2.VideoCapture("test_clip.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter("outputs/annotated.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

#MAX_FRAMES = 100  # for 100 frames test only

with open("outputs/positions.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["frame", "tracker_id", "x_center", "y_center", "confidence", "class"])

    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, verbose=False)[0]
        detections = sv.Detections.from_ultralytics(results)
        detections = detections[detections.class_id == 0]
        detections = tracker.update_with_detections(detections)

        for i in range(len(detections)):
            x1, y1, x2, y2 = detections.xyxy[i]
            writer.writerow([
                frame_idx,
                int(detections.tracker_id[i]),
                round(float((x1+x2)/2), 2),
                round(float((y1+y2)/2), 2),
                round(float(detections.confidence[i]), 3),
                int(detections.class_id[i])
            ])

        labels = [f"#{tid}" for tid in detections.tracker_id]
        frame = box_annotator.annotate(frame, detections)
        frame = label_annotator.annotate(frame, detections, labels)
        out.write(frame)

        if frame_idx % 10 == 0:
            print(f"Frame {frame_idx}: {len(detections)} players detected")

cap.release()
out.release()
print("Done!")

In [ ]:
#problem couldnt detect players 
import pandas as pd
df = pd.read_csv("outputs/positions.csv")
print(df.head(20))
print(f"\nTotal detections: {len(df)}")
print(f"Unique players tracked: {df['tracker_id'].nunique()}")
print(f"Frames processed: {df['frame'].nunique()}")

In [ ]:
#checking bytes ret and frame shape
import cv2, os
print("Size:", os.path.getsize("test_clip.mp4"), "bytes")
cap = cv2.VideoCapture("test_clip.mp4")
ret, frame = cap.read()
print("ret:", ret)
print("frame shape:", frame.shape if frame is not None else "None")
cap.release()

In [ ]:
#retrying with new file and checking 
import cv2, os

print("File exists:", os.path.exists("test_clip.mp4"))
print("File size:", os.path.getsize("test_clip.mp4"), "bytes")

cap = cv2.VideoCapture("test_clip.mp4")
print("Cap opened:", cap.isOpened())
print("Total frames:", cap.get(cv2.CAP_PROP_FRAME_COUNT))
ret, frame = cap.read()
print("ret:", ret)
print("frame:", frame)
cap.release()

In [ ]:
#extracting in 720p and mp4 format
!rm test_clip.mp4
!yt-dlp "https://youtu.be/9V2guLT3S14" -f "bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]" --merge-output-format mp4 -o test_clip.mp4 --no-part

In [ ]:
#similar
!rm -f test_clip.mp4
!yt-dlp "https://youtu.be/9V2guLT3S14" -f "bestvideo[height<=720][vcodec^=avc]+bestaudio[ext=m4a]" --merge-output-format mp4 -o test_clip.mp4

In [ ]:
#checking frame shape
import cv2
cap = cv2.VideoCapture("test_clip.mp4")
ret, frame = cap.read()
print("ret:", ret)
print("frame shape:", frame.shape if frame is not None else "None")
cap.release()

In [ ]:
#detecting players at every 50 frame and trying to get the best frame for team assigning
import cv2
import csv
import supervision as sv
from ultralytics import YOLO
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)

model = YOLO("yolov8x.pt")
tracker = sv.ByteTrack()
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

cap = cv2.VideoCapture("test_clip.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter("outputs/annotated.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

with open("outputs/positions.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["frame", "tracker_id", "x_center", "y_center", "confidence", "class"])

    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, verbose=False)[0]
        detections = sv.Detections.from_ultralytics(results)
        detections = detections[detections.class_id == 0]
        detections = tracker.update_with_detections(detections)

        for i in range(len(detections)):
            x1, y1, x2, y2 = detections.xyxy[i]
            writer.writerow([
                frame_idx,
                int(detections.tracker_id[i]),
                round(float((x1+x2)/2), 2),
                round(float((y1+y2)/2), 2),
                round(float(detections.confidence[i]), 3),
                int(detections.class_id[i])
            ])

        labels = [f"#{tid}" for tid in detections.tracker_id]
        frame = box_annotator.annotate(frame, detections)
        frame = label_annotator.annotate(frame, detections, labels)
        out.write(frame)

        if frame_idx % 50 == 0:
            print(f"Frame {frame_idx}: {len(detections)} players detected")

        frame_idx += 1

cap.release()
out.release()
print(f"\nDone! Total frames: {frame_idx}")

In [ ]:
#tracking players
import pandas as pd
df = pd.read_csv("outputs/positions.csv")
print(df.head(20))
print(f"\nTotal detections: {len(df)}")
print(f"Unique players tracked: {df['tracker_id'].nunique()}")
print(f"Frames processed: {df['frame'].nunique()}")

In [ ]:
# checking multiple frames finding the best frame
import cv2
from ultralytics import YOLO
import supervision as sv

model = YOLO("yolov8x.pt")
cap = cv2.VideoCapture("test_clip.mp4")

best_frame_idx = 0
best_count = 0

# har 200 frames pe check karo
for frame_num in range(0, 6214, 200):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
    ret, frame = cap.read()
    if not ret:
        continue
    results = model(frame, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(results)
    detections = detections[detections.class_id == 0]
    print(f"Frame {frame_num}: {len(detections)} players")
    if len(detections) > best_count:
        best_count = len(detections)
        best_frame_idx = frame_num

cap.release()
print(f"\nBest frame: {best_frame_idx} with {best_count} players")

In [ ]:
#returning all players we see in the best frame we got ie 4800
import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

model = YOLO("yolov8x.pt")
cap = cv2.VideoCapture("test_clip.mp4")

# frame pe jao : match shuru ho chuka hoga
cap.set(cv2.CAP_PROP_POS_FRAMES, 4800)
ret, frame = cap.read()
cap.release()

results = model(frame, verbose=False)[0]

import supervision as sv
detections = sv.Detections.from_ultralytics(results)
detections = detections[detections.class_id == 0]

print(f"Players detected: {len(detections)}")

# har player ka upper-half crop dikho
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
axes = axes.flatten()

for i, (x1, y1, x2, y2) in enumerate(detections.xyxy[:18]):
    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
    mid_y = int((y1 + y2) / 2)
    crop = frame[y1:mid_y, x1:x2]  # upper half = jersey
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    axes[i].imshow(crop_rgb)
    axes[i].axis("off")
    axes[i].set_title(f"P{i+1}")

plt.tight_layout()
plt.savefig("outputs/crops.png")
plt.show()
print("Crops saved to outputs/crops.png")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from ultralytics import YOLO
import supervision as sv

model = YOLO("yolov8x.pt")
cap = cv2.VideoCapture("test_clip.mp4")
cap.set(cv2.CAP_PROP_POS_FRAMES, best_frame_idx)
ret, frame = cap.read()
cap.release()

results = model(frame, verbose=False)[0]
detections = sv.Detections.from_ultralytics(results)
detections = detections[detections.class_id == 0]

# player jersey color (upper half mean RGB)
colors = []
for x1, y1, x2, y2 in detections.xyxy:
    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
    mid_y = int((y1 + y2) / 2)
    crop = frame[y1:mid_y, x1:x2]
    if crop.size == 0:
        colors.append([0, 0, 0])
        continue
    mean_color = crop.reshape(-1, 3).mean(axis=0)  # BGR
    colors.append(mean_color)

colors = np.array(colors)

# K-means : 2 teams + 1 referee/other
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(colors)

# visualize
fig, axes = plt.subplots(3, 8, figsize=(20, 8))
axes = axes.flatten()

team_colors = ['red', 'blue', 'green']  # 0=team A, 1=team B, 2=other
for i, (x1, y1, x2, y2) in enumerate(detections.xyxy[:24]):
    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
    mid_y = int((y1 + y2) / 2)
    crop = frame[y1:mid_y, x1:x2]
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    axes[i].imshow(crop_rgb)
    axes[i].axis("off")
    axes[i].set_title(f"Team {labels[i]}", color=team_colors[labels[i]])

plt.tight_layout()
plt.savefig("outputs/team_assignment.png")
plt.show()

print("\nCluster centers (BGR):")
for i, center in enumerate(kmeans.cluster_centers_):
    print(f"  Cluster {i}: B={int(center[0])}, G={int(center[1])}, R={int(center[2])}")
print(f"\nTeam distribution: {np.bincount(labels)}")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from ultralytics import YOLO
import supervision as sv

model = YOLO("yolov8x.pt")
cap = cv2.VideoCapture("test_clip.mp4")
cap.set(cv2.CAP_PROP_POS_FRAMES, best_frame_idx)
ret, frame = cap.read()
cap.release()

results = model(frame, verbose=False)[0]
detections = sv.Detections.from_ultralytics(results)
detections = detections[detections.class_id == 0]

# Step 1 - jersey colors extraction
colors = []
valid_indices = []

for idx, (x1, y1, x2, y2) in enumerate(detections.xyxy):
    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
    mid_y = int((y1 + y2) / 2)
    crop = frame[y1:mid_y, x1:x2]
    if crop.size == 0:
        continue
    mean_color = crop.reshape(-1, 3).mean(axis=0)
    colors.append(mean_color)
    valid_indices.append(idx)

colors = np.array(colors)

# Step 2 - K-means 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(colors)
counts = np.bincount(labels)

print(f"Cluster sizes: {counts}")

# Step 3 - assignment
# 2 biggest clusters = Team A and Team B
# smallest cluster = Others (referee, managers, staff)
sorted_clusters = np.argsort(counts)[::-1]  # descending order by size

team_map = {}
team_map[sorted_clusters[0]] = 0  # Team A (biggest)
team_map[sorted_clusters[1]] = 1  # Team B (second biggest)
team_map[sorted_clusters[2]] = 2  # Others (smallest)

final_labels = np.array([team_map[l] for l in labels])

print(f"Team A: {np.sum(final_labels==0)} players")
print(f"Team B: {np.sum(final_labels==1)} players")
print(f"Others: {np.sum(final_labels==2)} players (refs/managers)")

# Step 4 — visualize
fig, axes = plt.subplots(3, 8, figsize=(20, 8))
axes = axes.flatten()

colors_viz = ['red', 'blue', 'gray']
labels_viz = ['Team A', 'Team B', 'Other']

for plot_idx, (det_idx, label) in enumerate(zip(valid_indices, final_labels)):
    if plot_idx >= 24:
        break
    x1, y1, x2, y2 = detections.xyxy[det_idx]
    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
    mid_y = int((y1 + y2) / 2)
    crop = frame[y1:mid_y, x1:x2]
    if crop.size == 0:
        continue
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    axes[plot_idx].imshow(crop_rgb)
    axes[plot_idx].axis("off")
    axes[plot_idx].set_title(labels_viz[label], color=colors_viz[label])

# empty axes hide karo
for j in range(plot_idx+1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig("outputs/team_assignment_v3.png")
plt.show()

from google.colab import files
files.download("outputs/team_assignment_v3.png")

In [ ]:
# outputs/positions.csv add in team column 
import pandas as pd
import numpy as np

print("Team assignment working!")
print(f"Team A: {np.sum(final_labels==0)} | Team B: {np.sum(final_labels==1)} | Others: {np.sum(final_labels==2)}")